In [1]:
# =========================================
# BALL CATCH TEST (FULL END-TO-END)
# =========================================

import cv2
import numpy as np
import mediapipe as mp

In [2]:
# -------------------------------
# MEDIAPIPE SETUP
# -------------------------------
mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence=0.6)

mpHands = mp.solutions.hands
hands = mpHands.Hands(min_detection_confidence=0.6)

mpDraw = mp.solutions.drawing_utils

In [3]:
# -------------------------------
# UTILS
# -------------------------------
def safe_mean(x): return float(np.mean(x)) if len(x) else 0.0
def safe_std(x): return float(np.std(x)) if len(x) else 0.0

In [4]:
# -------------------------------
# BALL DETECTION (COLOR BASED)
# -------------------------------
def detect_ball(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower = np.array([30, 100, 100])   # adjust for ball color
    upper = np.array([50, 255, 255])

    mask = cv2.inRange(hsv, lower, upper)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        c = max(contours, key=cv2.contourArea)
        (x, y), r = cv2.minEnclosingCircle(c)
        if r > 5:
            return int(x), int(y), int(r)

    return None

In [5]:
# -------------------------------
# SCORING FUNCTION
# -------------------------------
def compute_final_ball_score(catches,eye_score,arm_score,bilateral_score,timing_score,zone_score,major_errors,age_group):

    # SUCCESS SCORE
    if age_group == "6-7":
        if catches >= 8: success = 5
        elif catches >= 6: success = 4
        elif catches >= 4: success = 3
        elif catches >= 2: success = 2
        else: success = 1
    else:
        if catches >= 9: success = 5
        elif catches >= 7: success = 4
        elif catches >= 5: success = 3
        elif catches >= 3: success = 2
        else: success = 1

    # FORM SCORE
    form = round((eye_score + arm_score + bilateral_score + timing_score + zone_score) / 5)

    # PENALTY
    if major_errors >= 6:
        form = max(1, form - 2)
    elif major_errors >= 3:
        form = max(1, form - 1)

    final = round((success + form) / 2)

    return final, success, form

In [6]:
def compute_eye_score(eye_errors):

    mean_err = safe_mean(eye_errors)
    std_err = safe_std(eye_errors)

    # consistency matters as well
    if mean_err < 40 and std_err < 15:
        return 5   # continuous tracking

    elif mean_err < 60 and std_err < 25:
        return 4   # minor deviation

    elif mean_err < 80:
        return 3   # intermittent tracking

    elif mean_err < 120:
        return 2   # frequent loss

    else:
        return 1   # no tracking

In [7]:
def compute_arm_score(arm_errors):

    mean_err = safe_mean(arm_errors)

    if mean_err < 60:
        return 5   # ideal chest-level ready position

    elif mean_err < 90:
        return 4   # slight deviation

    elif mean_err < 120:
        return 3   # inconsistent

    elif mean_err < 160:
        return 2   # poor positioning

    else:
        return 1   # no preparation

In [8]:
def compute_bilateral_score(bilateral_errors):

    mean_err = safe_mean(bilateral_errors)
    std_err = safe_std(bilateral_errors)

    if mean_err < 30 and std_err < 15:
        return 5   # symmetrical

    elif mean_err < 50:
        return 4   # minor mismatch

    elif mean_err < 80:
        return 3   # moderate mismatch

    elif mean_err < 120:
        return 2   # poor coordination

    else:
        return 1   # one-hand dominant

In [9]:
def compute_timing_score(timing_errors):

    mean_err = safe_mean(timing_errors)
    std_err = safe_std(timing_errors)

    if mean_err < 8 and std_err < 5:
        return 5   # perfect anticipation

    elif mean_err < 12:
        return 4   # slight delay

    elif mean_err < 20:
        return 3   # inconsistent timing

    elif mean_err < 30:
        return 2   # mostly mistimed

    else:
        return 1   # no anticipation

In [10]:
def compute_zone_score(zone_scores):

    if not zone_scores:
        return 1

    avg_zone = safe_mean(zone_scores)

    if avg_zone >= 4.5:
        return 5   # chest level

    elif avg_zone >= 3.5:
        return 4   # upper torso

    elif avg_zone >= 2.5:
        return 3   # waist level

    elif avg_zone >= 1.5:
        return 2   # below waist

    else:
        return 1   # knee/drop level

In [21]:
# MAIN FUNCTION
# -------------------------------
def ball_catch_test(path, age_group="6-7"):

    # cap = cv2.VideoCapture(path)
    cap = cv2.VideoCapture(0)
    
    catches = 0
    major_errors = 0

    prev_ball_y = None
    throw_active = False

    # store metrics
    eye_errors = []
    arm_errors = []
    bilateral_errors = []
    timing_errors = []
    zone_scores = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape

        ball = detect_ball(frame)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pose_res = pose.process(rgb)
        hand_res = hands.process(rgb)

        if pose_res.pose_landmarks:

            lms = pose_res.pose_landmarks.landmark

            # key points
            l_wri = (lms[15].x * w, lms[15].y * h)
            r_wri = (lms[16].x * w, lms[16].y * h)
            l_sh = (lms[11].x * w, lms[11].y * h)
            r_sh = (lms[12].x * w, lms[12].y * h)
            l_knee = (lms[25].x * w, lms[25].y * h)
            r_knee = (lms[26].x * w, lms[26].y * h)
            nose = (lms[0].x * w, lms[0].y * h)

            if ball:
                bx, by, br = ball
                cv2.circle(frame, (bx, by), br, (0,255,0), 2)

                # detect throw
                if prev_ball_y is not None and by < prev_ball_y:
                    throw_active = True

                if throw_active:

                    # ---------------- EYE TRACKING ----------------
                    eye_err = abs(nose[0] - bx)
                    eye_errors.append(eye_err)

                    # ---------------- ARM POSITION ----------------
                    arm_err = abs(l_wri[1] - l_sh[1]) + abs(r_wri[1] - r_sh[1])
                    arm_errors.append(arm_err)

                    # ---------------- BILATERAL ----------------
                    bil_err = abs(l_wri[1] - r_wri[1])
                    bilateral_errors.append(bil_err)

                    # ---------------- TIMING ----------------
                    if prev_ball_y is not None:
                        timing_err = abs(by - prev_ball_y)
                        timing_errors.append(timing_err)

                    # ---------------- ZONE + CATCH ----------------
                    knee_level = (l_knee[1] + r_knee[1]) / 2
                    shoulder_level = (l_sh[1] + r_sh[1]) / 2

                    # knee rule → miss
                    if by > knee_level:
                        zone_scores.append(1)
                        major_errors += 1
                        throw_active = False

                    else:
                        # distance to hands
                        d_l = np.linalg.norm(np.array([bx,by]) - np.array(l_wri))
                        d_r = np.linalg.norm(np.array([bx,by]) - np.array(r_wri))

                        if d_l < 50 and d_r < 50:
                            catches += 1

                            # zone scoring
                            if by < shoulder_level:
                                zone_scores.append(5)
                            elif by < shoulder_level + 50:
                                zone_scores.append(4)
                            elif by < knee_level:
                                zone_scores.append(3)
                            else:
                                zone_scores.append(2)

                            throw_active = False

                prev_ball_y = by

        # draw pose
        if pose_res.pose_landmarks:
            mpDraw.draw_landmarks(frame, pose_res.pose_landmarks, mpPose.POSE_CONNECTIONS)

        if hand_res.multi_hand_landmarks:
            for hnd in hand_res.multi_hand_landmarks:
                mpDraw.draw_landmarks(frame, hnd, mpHands.HAND_CONNECTIONS)

        cv2.imshow("Ball Catch", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    # =========================
    # CONVERT METRICS → SCORES

    eye_score = compute_eye_score(eye_errors)
    arm_score = compute_arm_score(arm_errors)
    bilateral_score = compute_bilateral_score(bilateral_errors)
    timing_score = compute_timing_score(timing_errors)
    zone_score = compute_zone_score(zone_scores)

    # =========================
    # FINAL SCORE
    # =========================
    final, success, form = compute_final_ball_score(catches,eye_score,arm_score,bilateral_score,timing_score,zone_score,major_errors,age_group)

    # =========================
    # OUTPUT
    # =========================
    print("------ BALL CATCH RESULT ------")
    print(f"Catches: {catches}")
    print(f"Eye Score: {eye_score}")
    print(f"Arm Score: {arm_score}")
    print(f"Bilateral Score: {bilateral_score}")
    print(f"Timing Score: {timing_score}")
    print(f"Zone Score: {zone_score}")
    print(f"Major Errors: {major_errors}")
    print(f"Success Score: {success}")
    print(f"Form Score: {form}")
    print(f"Final Score: {final}")

    return final

In [24]:
path = "data/shuttle_run.mp4"
ball_catch_test(path, age_group="6-7")

------ BALL CATCH RESULT ------
Catches: 0
Eye Score: 1
Arm Score: 1
Bilateral Score: 1
Timing Score: 1
Zone Score: 1
Major Errors: 0
Success Score: 1
Form Score: 1
Final Score: 1


1